# ПЗ 3 — OCR из потока кадров (субтитры из фильма)

**Задача:** из набора кадров (результат ПЗ 2) вырезать нижнюю полосу, где расположены субтитры, и распознать текст через EasyOCR.

**Стек:** `easyocr`, `opencv-python`, `pandas`

In [ ]:
!pip install easyocr opencv-python-headless pandas -q

In [ ]:
import os
import cv2
import easyocr
import pandas as pd
from tqdm.notebook import tqdm

FRAMES_DIR  = '/content/outputs/frames'   # папка с кадрами из ПЗ 2
OUTPUT_CSV  = '/content/outputs/ocr_results/subtitles.csv'
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

SUBTITLE_ROW_FRACTION = 0.15  # нижние 15% кадра — зона субтитров
reader = easyocr.Reader(['ru', 'en'], gpu=True)
print('EasyOCR инициализирован')

In [ ]:
frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
results = []

for fname in tqdm(frames, desc='OCR'):
    img = cv2.imread(os.path.join(FRAMES_DIR, fname))
    h, w = img.shape[:2]
    crop_y = int(h * (1 - SUBTITLE_ROW_FRACTION))
    subtitle_region = img[crop_y:, :]
    gray = cv2.cvtColor(subtitle_region, cv2.COLOR_BGR2GRAY)

    ocr_out = reader.readtext(gray, detail=1)
    for (bbox, text, prob) in ocr_out:
        text = text.strip()
        if text and prob > 0.4:
            results.append({'frame': fname, 'text': text, 'confidence': round(prob, 3)})

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'Распознано строк: {len(df)}')
df.head(10)

In [ ]:
# Уникальные фразы (предварительная дедупликация)
unique_texts = df['text'].drop_duplicates().reset_index(drop=True)
print(f'Уникальных фраз: {len(unique_texts)}')
print(unique_texts.head(20).to_string())